# 04 — Cost-Sensitive Analysis & Optimal Intervention Threshold

**Goal:** Convert model probabilities into a business decision. The default 0.5 threshold is a modeling convention, not a financial one. We want to find the threshold $\tau^*$ that maximizes net realized value of a retention policy on the test set.

**The decision:** for each customer with predicted churn probability $p$, intervene iff

$$p \cdot \text{success\_rate} \cdot \text{CLV} > \text{intervention\_cost}$$

**Three knobs the business owns (not the model):**
1. Intervention cost (per-customer $)
2. Intervention success rate (probability the play works on a true churner)
3. CLV horizon (how many months of MRR we credit to a save)

All three become parameters; we scan threshold for one set of assumptions, then do sensitivity analysis.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.cost_analysis import (
    CostAssumptions,
    optimal_threshold,
    policy_value,
    sweep_thresholds,
)
from src.data_loader import load_clean
from src.evaluation import plot_confusion
from src.features import make_features, train_test_split_scaled
from src.model import train_xgboost

sns.set_theme(style="whitegrid")

## 1. Train the production model and score the test set

We use XGBoost (the Week 1 winner). We also keep `MonthlyCharges` aligned with the test split so we can compute per-customer CLV downstream.

In [ ]:
df = load_clean()
X, y = make_features(df)
X_train, X_test, y_train, y_test = train_test_split_scaled(X, y, scale=True)

# Recover unscaled MonthlyCharges aligned with X_test rows.
# We re-derive it from the original df using the test indices.
monthly_charges_test = df.loc[X_test.index, "MonthlyCharges"].values

model = train_xgboost(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
print(f"Test set: {len(y_test)} customers, {y_test.sum()} churners ({y_test.mean():.1%})")

## 2. Set business assumptions

These are placeholders pending real values from the business. They live in one cell so sensitivity analysis is trivial later.

In [ ]:
assumptions = CostAssumptions(
    intervention_cost=50.0,    # $ per targeted customer (call + small discount)
    success_rate=0.30,         # 30% of true churners are saved by the play
    clv_horizon_months=12,     # credit 12 months of MRR per save
)
assumptions

## 3. Naive baselines for context

Before optimizing, anchor on three trivial policies so the optimization gain is interpretable:

- **Do nothing** (threshold = 1.01): targets no one, loses revenue from every churner.
- **Treat everyone** (threshold = 0.0): pays intervention cost on the whole base.
- **Default 0.5 threshold**: the modeling convention.

In [ ]:
baselines = pd.DataFrame([
    {"policy": "do_nothing", **policy_value(y_test, y_proba, monthly_charges_test, 1.01, assumptions)},
    {"policy": "treat_everyone", **policy_value(y_test, y_proba, monthly_charges_test, 0.0, assumptions)},
    {"policy": "threshold_0.5", **policy_value(y_test, y_proba, monthly_charges_test, 0.5, assumptions)},
]).set_index("policy")
baselines.round(0)

## 4. Sweep thresholds and find the optimum

In [ ]:
sweep = sweep_thresholds(y_test, y_proba, monthly_charges_test, assumptions)
best = optimal_threshold(sweep)
print(f"Optimal threshold: {best['threshold']:.2f}")
print(f"  Targets:        {int(best['n_targeted'])} customers")
print(f"  Net value:      ${best['net_value']:,.0f}")
print(f"  Saved revenue:  ${best['saved_revenue']:,.0f}")
print(f"  Lost revenue:   ${best['lost_revenue']:,.0f}")
print(f"  Intervention $: ${best['intervention_costs']:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sweep["threshold"], sweep["net_value"], color="steelblue", linewidth=2)
ax.axvline(best["threshold"], color="red", linestyle="--", alpha=0.7,
           label=f"optimum = {best['threshold']:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.7, label="default 0.5")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("intervention threshold (predicted P(churn))")
ax.set_ylabel("net realized value ($)")
ax.set_title("Expected-value curve across thresholds")
ax.legend()
plt.show()

## 5. Confusion matrix at the optimal threshold

Translate $\tau^*$ back to the operational reality: how many true churners do we catch, and how many non-churners do we waste interventions on?

In [ ]:
y_pred_opt = (y_proba >= best["threshold"]).astype(int)
plot_confusion(y_test, y_pred_opt)
plt.title(f"Confusion matrix at threshold = {best['threshold']:.2f}")
plt.show()

## 6. Sensitivity analysis — how does $\tau^*$ move with our assumptions?

If a stakeholder pushes back on our 30% success-rate assumption, we want to know how much that would change our recommendation. Same for intervention cost. We sweep both and plot the optimal threshold surface.

In [ ]:
cost_grid = [25, 50, 75, 100, 150]
rate_grid = [0.15, 0.20, 0.30, 0.40, 0.50]

rows = []
for c in cost_grid:
    for r in rate_grid:
        a = CostAssumptions(intervention_cost=c, success_rate=r,
                            clv_horizon_months=assumptions.clv_horizon_months)
        s = sweep_thresholds(y_test, y_proba, monthly_charges_test, a)
        b = optimal_threshold(s)
        rows.append({
            "intervention_cost": c,
            "success_rate": r,
            "optimal_threshold": b["threshold"],
            "net_value": b["net_value"],
        })

sensitivity = pd.DataFrame(rows)
pivot_threshold = sensitivity.pivot(index="intervention_cost", columns="success_rate",
                                    values="optimal_threshold")
pivot_value = sensitivity.pivot(index="intervention_cost", columns="success_rate",
                                values="net_value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(pivot_threshold, annot=True, fmt=".2f", cmap="viridis", ax=axes[0])
axes[0].set_title("Optimal threshold $\\tau^*$")
sns.heatmap(pivot_value / 1000, annot=True, fmt=".0f", cmap="YlGn", ax=axes[1])
axes[1].set_title("Net value at $\\tau^*$ ($K)")
plt.tight_layout()
plt.show()

**Reading the heatmaps:**
- *Higher intervention cost* → threshold rises (be choosier about who we target).
- *Higher success rate* → threshold falls (interventions are more valuable, target more aggressively).
- The **net-value** heatmap (right) shows how much the program is worth across the assumption space — useful for stakeholder conversations.

## 7. Handoff to Week 3

Outputs that downstream notebooks will reuse:
- **Optimal threshold $\tau^*$** — feeds the SQL view `at_risk_cohorts` (Week 3) so the analyst-facing query mirrors the model's recommendation.
- **Targeted-customer list** at $\tau^*$ — exported to `data/at_risk_test.csv` (next cell) for the Tableau dashboard.
- **Sensitivity table** — a stakeholder-facing exhibit for the README's Headline Results section.

In [ ]:
# Export the targeted cohort for downstream reuse
at_risk = pd.DataFrame({
    "customerID": df.loc[X_test.index, "customerID"].values,
    "churn_probability": y_proba,
    "monthly_charges": monthly_charges_test,
    "actual_churn": y_test.values,
    "targeted": y_pred_opt,
})
out_path = Path("../data/at_risk_test.csv")
at_risk.to_csv(out_path, index=False)
print(f"Wrote {out_path.resolve()} ({len(at_risk)} rows, {at_risk['targeted'].sum()} targeted)")